# 📓 Notebook: Applied Deep Learning & The Road to Foundation Models

**Bootcamp AI Engineering - ReWork Academy**

Notebook ini dirancang sebagai pendamping interaktif untuk Sesi 5. Kita akan beralih dari teori menuju praktik langsung, melihat bagaimana mesin belajar dari nol (skala kecil), dan bagaimana AI Engineer di industri menggunakan *Pre-trained Models* (skala besar) untuk mempercepat pekerjaan.

## 🔗 Bagian 1: Visualisasi Konsep (Referensi)
*(Bagian ini tidak membutuhkan kode, mentor akan mendemonstrasikan via browser)*

*   **1.1 Neural Network Architecture:** [TensorFlow Playground](https://playground.tensorflow.org/)
*   **1.2 Text Embeddings & Ruang Vektor:** [Embedding Projector](https://projector.tensorflow.org/)

## ⚙️ Bagian 2: Demo - Bagaimana Mesin Belajar (Training dari Nol)
Di sini kita akan melihat visualisasi proses *Gradient Descent* dan *Backpropagation* menggunakan PyTorch. Kita menggunakan *toy model* sederhana untuk memprediksi pola angka, agar kita bisa melihat metrik **Loss** terus menurun.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets

# 1. Siapkan Dataset Iris
iris = datasets.load_iris()
X_train = torch.tensor(iris.data, dtype=torch.float32)
y_train = torch.tensor(iris.target, dtype=torch.long) # Use Long for CrossEntropyLoss

# 2. Definisikan Neural Network untuk Iris Dataset
# Iris memiliki 4 fitur input dan 3 kelas output
class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        # Input layer (4 features from Iris) to hidden layer (10 features)
        self.hidden = nn.Linear(4, 5)
        # Activation function for the hidden layer
        self.relu = nn.ReLU()

        # Hidden layer (10 features) to output layer (3 classes for Iris)
        self.output = nn.Linear(10, 3)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

model = ToyModel()

# 3. Definisikan Loss Function (Error) dan Optimizer (Cara belajar)
# Gunakan CrossEntropyLoss untuk klasifikasi multi-kelas
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01) # Stochastic Gradient Descent

# 4. Proses Training (Looping Epoch)
epochs = 1000
print("Memulai Proses Training untuk Iris Dataset...\n")
for epoch in range(epochs):
    # Forward pass: prediksi
    predictions = model(X_train)

    # Hitung error (Loss)
    loss = criterion(predictions, y_train)

    # Backward pass: perbaiki bobot (Backpropagation)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress setiap 20 epoch
    if (epoch + 1) % 20 == 0:
        # Untuk klasifikasi, loss bisa sangat kecil, tapi kita juga bisa melihat akurasi
        _, predicted = torch.max(predictions.data, 1)
        accuracy = (predicted == y_train).sum().item() / y_train.size(0)
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')

print("\nTraining Selesai! Model telah belajar mengklasifikasikan Iris Dataset.")

Memulai Proses Training untuk Iris Dataset...



RuntimeError: mat1 and mat2 shapes cannot be multiplied (150x5 and 10x3)

In [5]:
iris.data

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

## 🚀 Bagian 3: Praktik Terpadu - Sentiment Analysis (Inference)
Melatih model seperti GPT atau BERT dari nol sangat mahal dan memakan waktu. Praktik industri saat ini adalah menggunakan model yang sudah dilatih (Pre-trained). Kita akan menggunakan model `distilbert-base-uncased-finetuned-sst-2-english` dari Hugging Face untuk melakukan klasifikasi sentimen.

In [2]:
# Install library jika dijalankan di environment baru (Google Colab, dsb)
# !pip install torch transformers

In [3]:
from transformers import pipeline

# 1. Gunakan model RoBERTa yang mendukung 3 kelas: Positive, Negative, Neutral
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 2. Inisialisasi pipeline (top_k=None memastikan semua probabilitas ditampilkan)
sentiment_model = pipeline(
    "text-classification",
    model=model_name,
    tokenizer=model_name,
    top_k=None
)



config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [4]:
# Daftar teks ulasan yang akan dianalisis
reviews = {
    "Positif": "The product quality is absolutely amazing, highly recommended!",
    "Negatif": "I am very disappointed with the service, it was terribly slow.",
    "Ambigu": "The camera quality is the best ever, but the battery life is quite dissapointed.",
    "Sarkas": "Wow, I love this wonderful software update that broke my laptop. My child is now sad"
}

# Melakukan inference untuk setiap teks
for category, text in reviews.items():
    print(f"--- Kategori: {category} ---")
    print(f"Teks  : '{text}'")

    # Eksekusi Model
    hasil = sentiment_model(text)[0]

    # Menampilkan hasil probabilitas
    for label_score in hasil:
        # Mengubah kapitalisasi label agar lebih rapi (misal: 'positive' -> 'POSITIVE')
        label = str(label_score['label']).upper()
        score = label_score['score']
        print(f"Probabilitas [{label}]: {score:.4f} ({score * 100:.2f}%)")
    print("\n")

--- Kategori: Positif ---
Teks  : 'The product quality is absolutely amazing, highly recommended!'
Probabilitas [POSITIVE]: 0.9834 (98.34%)
Probabilitas [NEUTRAL]: 0.0102 (1.02%)
Probabilitas [NEGATIVE]: 0.0064 (0.64%)


--- Kategori: Negatif ---
Teks  : 'I am very disappointed with the service, it was terribly slow.'
Probabilitas [NEGATIVE]: 0.9438 (94.38%)
Probabilitas [NEUTRAL]: 0.0499 (4.99%)
Probabilitas [POSITIVE]: 0.0063 (0.63%)


--- Kategori: Ambigu ---
Teks  : 'The camera quality is the best ever, but the battery life is quite dissapointed.'
Probabilitas [NEGATIVE]: 0.4129 (41.29%)
Probabilitas [POSITIVE]: 0.3226 (32.26%)
Probabilitas [NEUTRAL]: 0.2644 (26.44%)


--- Kategori: Sarkas ---
Teks  : 'Wow, I love this wonderful software update that broke my laptop. My child is now sad'
Probabilitas [POSITIVE]: 0.9062 (90.62%)
Probabilitas [NEUTRAL]: 0.0499 (4.99%)
Probabilitas [NEGATIVE]: 0.0440 (4.40%)




### Model lain

In [7]:
from transformers import pipeline

# Memuat pipeline klasifikasi teks dengan model ringan yang direkomendasikan ToR
print("Mengunduh/Memuat model dari cache...")
sentiment_model = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    top_k=None # Menampilkan semua probabilitas skor, tidak hanya hasil akhir
)
print("Model siap digunakan!")

Mengunduh/Memuat model dari cache...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model siap digunakan!


In [8]:
# Daftar teks ulasan yang akan dianalisis
reviews = {
    "Positif": "The product quality is absolutely amazing, highly recommended!",
    "Negatif": "I am very disappointed with the service, it was terribly slow.",
    "Ambigu": "The camera quality is the best ever, but the battery life is quite dissapointed.",
    "Sarkas": "Wow, I absolutely love and adore this wonderful software update that freezes my laptop."
}

# Melakukan inference untuk setiap teks
for category, text in reviews.items():
    print(f"--- Kategori: {category} ---")
    print(f"Teks  : '{text}'")

    # Eksekusi Model
    hasil = sentiment_model(text)[0]

    # Menampilkan hasil probabilitas
    for label_score in hasil:
        label = label_score['label']
        score = label_score['score']
        print(f"Probabilitas [{label}]: {score:.4f} ({score * 100:.2f}%)")
    print("\n")

--- Kategori: Positif ---
Teks  : 'The product quality is absolutely amazing, highly recommended!'
Probabilitas [POSITIVE]: 0.9999 (99.99%)
Probabilitas [NEGATIVE]: 0.0001 (0.01%)


--- Kategori: Negatif ---
Teks  : 'I am very disappointed with the service, it was terribly slow.'
Probabilitas [NEGATIVE]: 0.9998 (99.98%)
Probabilitas [POSITIVE]: 0.0002 (0.02%)


--- Kategori: Ambigu ---
Teks  : 'The camera quality is the best ever, but the battery life is quite dissapointed.'
Probabilitas [NEGATIVE]: 0.9971 (99.71%)
Probabilitas [POSITIVE]: 0.0029 (0.29%)


--- Kategori: Sarkas ---
Teks  : 'Wow, I absolutely love and adore this wonderful software update that freezes my laptop.'
Probabilitas [POSITIVE]: 0.9997 (99.97%)
Probabilitas [NEGATIVE]: 0.0003 (0.03%)




### 💡 Diskusi Reflektif (Pertanyaan untuk Peserta)
1.  Perhatikan probabilitas pada teks **Ambigu**. Mengapa skornya tidak seyakin (mendekati 50:50) teks positif/negatif yang jelas?
2.  Perhatikan teks **Sarkas**. Apakah model berhasil menangkap maksud asli pengguna, atau justru terkecoh oleh kata-kata seperti *"great"* dan *"Just what I needed"*? Mengapa hal ini bisa menjadi masalah di sistem produksi nyata?